# 🥉 Bronze Layer – Raw Ingestion
**Chicago Crime Dataset – ETL Workflow Automation**

This notebook ingests raw CSV data from ADLS Gen2 into the Bronze layer with minimal transformation.
It is fully parameterized and called by Azure Data Factory.

In [0]:
# ─────────────────────────────────────────────
# 1. PARAMETERS  (overridden by ADF at runtime)
# ─────────────────────────────────────────────
dbutils.widgets.text("source_container",   "raw",                      "Source Container")
dbutils.widgets.text("source_path",        "chicago_crime/",           "Source Path (CSV folder)")
dbutils.widgets.text("bronze_container",   "bronze",                   "Bronze Container")
dbutils.widgets.text("bronze_path",        "chicago_crime/",           "Bronze Output Path")
dbutils.widgets.text("storage_account",    "etlworkflowautomation",   "ADLS Storage Account")
dbutils.widgets.text("run_date",           "",                         "Run Date (YYYY-MM-DD, blank=today)")
dbutils.widgets.text("pipeline_run_id",    "manual",                   "ADF Pipeline Run ID")

SOURCE_CONTAINER  = dbutils.widgets.get("source_container")
SOURCE_PATH       = dbutils.widgets.get("source_path")
BRONZE_CONTAINER  = dbutils.widgets.get("bronze_container")
BRONZE_PATH       = dbutils.widgets.get("bronze_path")
STORAGE_ACCOUNT   = dbutils.widgets.get("storage_account")
RUN_DATE          = dbutils.widgets.get("run_date")
PIPELINE_RUN_ID   = dbutils.widgets.get("pipeline_run_id")

print(f"Source  : abfss://{SOURCE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{SOURCE_PATH}")
print(f"Bronze  : abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{BRONZE_PATH}")

Source : abfss://raw@etlworkflowautomation.dfs.core.windows.net/chicago_crime/
Bronze : abfss://bronze@etlworkflowautomation.dfs.core.windows.net/chicago_crime/

In [0]:
# ─────────────────────────────────────────────
# 2. IMPORTS & SPARK SESSION
# ─────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, input_file_name
from datetime import datetime, date
import traceback

spark = SparkSession.builder.appName("Bronze_Chicago_Crime_Ingestion").getOrCreate()

RUN_DATE = RUN_DATE if RUN_DATE else str(date.today())
START_TIME = datetime.now()
print(f"Run date : {RUN_DATE}  |  Pipeline run ID : {PIPELINE_RUN_ID}")

Run date : 2026-05-20 | Pipeline run ID : manual

In [0]:
# ─────────────────────────────────────────────
# 3. MOUNT / CONFIGURE ADLS ACCESS
#    (Uses Databricks secret scope; set up once)
# ─────────────────────────────────────────────
SECRET_SCOPE = "etlworkflowautomation"         
SECRET_KEY   = "*********************************"   

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SECRET_KEY
)

In [0]:
# ─────────────────────────────────────────────
# 4. DEFINE SCHEMA (enforce on read for safety)
#
#    Column names match the EXACT CSV headers from
#    the Kaggle Chicago Crime dataset (22 columns).
#    All dates/coordinates kept as strings here;
#    proper types are cast in the Silver notebook.
# ─────────────────────────────────────────────
from pyspark.sql.types import (
    StructType, StructField,
    StringType, BooleanType, LongType, IntegerType, DoubleType
)

CRIME_SCHEMA = StructType([
    # ── Identifiers ──────────────────────────────────────────────────────
    StructField("ID",                   LongType(),    True),  # unique record key
    StructField("Case Number",          StringType(),  True),  # CPD RD Number

    # ── Incident details ─────────────────────────────────────────────────
    StructField("Date",                 StringType(),  True),  # raw string → cast in Silver
    StructField("Block",                StringType(),  True),  # partially redacted address
    StructField("IUCR",                 StringType(),  True),  # Illinois UCR code
    StructField("Primary Type",         StringType(),  True),  # primary IUCR description
    StructField("Description",          StringType(),  True),  # secondary IUCR description
    StructField("Location Description", StringType(),  True),  # location category
    StructField("Arrest",               BooleanType(), True),  # was an arrest made
    StructField("Domestic",             BooleanType(), True),  # domestic-related flag

    # ── Geographic / administrative ──────────────────────────────────────
    StructField("Beat",                 IntegerType(), True),  # police beat
    StructField("District",             IntegerType(), True),  # police district
    StructField("Ward",                 IntegerType(), True),  # city council ward
    StructField("Community Area",       IntegerType(), True),  # community area number
    StructField("FBI Code",             StringType(),  True),  # FBI crime classification

    # ── Coordinates ──────────────────────────────────────────────────────
    StructField("X Coordinate",         DoubleType(),  True),  # Illinois State Plane Easting
    StructField("Y Coordinate",         DoubleType(),  True),  # Illinois State Plane Northing
    StructField("Year",                 IntegerType(), True),  # incident year
    StructField("Updated On",           StringType(),  True),  # record last updated (string → Silver)
    StructField("Latitude",             DoubleType(),  True),  # WGS84 latitude
    StructField("Longitude",            DoubleType(),  True),  # WGS84 longitude
    StructField("Location",             StringType(),  True),  # '(lat, lon)' composite string
])

# Column name → safe snake_case mapping used after read
# (Spark allows spaces in column names but downstream SQL is cleaner without them)
COLUMN_RENAME_MAP = {
    "ID":                   "id",
    "Case Number":          "case_number",
    "Date":                 "date",
    "Block":                "block",
    "IUCR":                 "iucr",
    "Primary Type":         "primary_type",
    "Description":          "description",
    "Location Description": "location_description",
    "Arrest":               "arrest",
    "Domestic":             "domestic",
    "Beat":                 "beat",
    "District":             "district",
    "Ward":                 "ward",
    "Community Area":       "community_area",
    "FBI Code":             "fbi_code",
    "X Coordinate":         "x_coordinate",
    "Y Coordinate":         "y_coordinate",
    "Year":                 "year",
    "Updated On":           "updated_on",
    "Latitude":             "latitude",
    "Longitude":            "longitude",
    "Location":             "location",
}

print(f"Schema defined: {len(CRIME_SCHEMA.fields)} fields")

Schema defined: 22 fields

In [0]:
# ─────────────────────────────────────────────
# 5. READ RAW CSV
# ─────────────────────────────────────────────
source_uri = f"wasbs://{SOURCE_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{SOURCE_PATH}"

try:
    df_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("badRecordsPath", f"{source_uri}_quarantine/")
        .schema(CRIME_SCHEMA)
        .csv(source_uri)
    )

    # Rename spaced CSV headers → snake_case for SQL compatibility
    for original, renamed in COLUMN_RENAME_MAP.items():
        if original in df_raw.columns:
            df_raw = df_raw.withColumnRenamed(original, renamed)

    raw_count = df_raw.count()
    print(f"Raw records read : {raw_count:,}")
    print(f"   Columns          : {df_raw.columns}")
except Exception as e:
    print(f"Read failed: {e}")
    traceback.print_exc()
    dbutils.notebook.exit(f"FAILED|0|{str(e)}")

Raw records read : 8,554,479
 Columns : ['id', 'case_number', 'date', 'block', 'iucr', 'primary_type', 'description', 'location_description', 'arrest', 'domestic', 'beat', 'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate', 'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude', 'location']

In [0]:
# ─────────────────────────────────────────────
# 6. ADD AUDIT METADATA COLUMNS
# ─────────────────────────────────────────────
df_bronze = (
    df_raw
    .withColumn("_ingested_at",      current_timestamp())
    .withColumn("_source_file",      input_file_name())
    .withColumn("_pipeline_run_id",  lit(PIPELINE_RUN_ID))
    .withColumn("_run_date",         lit(RUN_DATE))
)

df_bronze.printSchema()
df_bronze.show(5, truncate=True)

root
-- id: long (nullable = true)
-- case_number: string (nullable = true)
-- date: string (nullable = true)
-- block: string (nullable = true)
-- iucr: string (nullable = true)
-- primary_type: string (nullable = true)
-- description: string (nullable = true)
-- location_description: string (nullable = true)
-- arrest: boolean (nullable = true)
-- domestic: boolean (nullable = true)
-- beat: integer (nullable = true)
-- district: integer (nullable = true)
-- ward: integer (nullable = true)
-- community_area: integer (nullable = true)
-- fbi_code: string (nullable = true)
-- x_coordinate: double (nullable = true)
-- y_coordinate: double (nullable = true)
-- year: integer (nullable = true)
-- updated_on: string (nullable = true)
-- latitude: double (nullable = true)
-- longitude: double (nullable = true)
-- location: string (nullable = true)
-- _ingested_at: timestamp (nullable = false)
-- _source_file: string (nullable = false)
-- _pipeline_run_id: string (nullable = false)
-- _run_date: string (nullable = false)

+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+--------------------+--------------------+----------------+---------+
 id|case_number| date| block|iucr| primary_type| description|location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year| updated_on| latitude| longitude| location| _ingested_at| _source_file|_pipeline_run_id|_run_date|
+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+--------------------+--------------------+----------------+---------+
13311263| JG503434|07/29/2022 03:39:...| 023XX S TROY ST|1582|OFFENSE INVOLVING...| CHILD PORNOGRAPHY| RESIDENCE| true| false|1033| 10| 25| 30| 17| null| null|2022|04/18/2024 03:40:...| null| null| null|2026-05-20 08:32:...|wasbs://raw@etlwo...| manual| |
13053066| JG103252|01/03/2023 04:44:...|039XX W WASHINGTO...|2017| NARCOTICS|MANUFACTURE / DEL...| SIDEWALK| true| false|1122| 11| 28| 26| 18| null| null|2023|01/20/2024 03:41:...| null| null| null|2026-05-20 08:32:...|wasbs://raw@etlwo...| manual| |
12131221| JD327000|08/10/2020 09:45:...| 015XX N DAMEN AVE|0326| ROBBERY|AGGRAVATED VEHICU...| STREET| true| false|1424| 14| 1| 24| 03| 1162795.0| 1909900.0|2020|05/17/2025 03:40:...|41.908417822| -87.67740693|(41.908417822, -8...|2026-05-20 08:32:...|wasbs://raw@etlwo...| manual| |
11227634| JB147599|08/26/2017 10:00:...| 001XX W RANDOLPH ST|0281| CRIM SEXUAL ASSAULT| NON-AGGRAVATED| HOTEL/MOTEL| false| false| 122| 1| 42| 32| 02| null| null|2017|02/11/2018 03:57:...| null| null| null|2026-05-20 08:32:...|wasbs://raw@etlwo...| manual| |
13203321| JG415333|09/06/2023 05:00:...| 002XX N Wells st|1320| CRIMINAL DAMAGE| TO VEHICLE|PARKING LOT / GAR...| false| false| 122| 1| 42| 32| 14| 1174694.0| 1901831.0|2023|11/04/2023 03:40:...|41.886018055|-87.633937881|(41.886018055, -8...|2026-05-20 08:32:...|wasbs://raw@etlwo...| manual| |
+--------+-----------+--------------------+--------------------+----+--------------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------------------+--------------------+--------------------+----------------+---------+
only showing top 5 rows

In [0]:
# ─────────────────────────────────────────────
# 7. WRITE TO BRONZE LAYER (Delta, partitioned)
# ─────────────────────────────────────────────
bronze_uri = f"wasbs://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net/{BRONZE_PATH}"

try:
    (
        df_bronze
        .write
        .format("delta")
        .mode("append")               # append; ADF prevents duplicate runs via triggers
        .partitionBy("_run_date")
        .option("mergeSchema", "true")
        .save(bronze_uri)
    )
    print(f"Bronze write complete → {bronze_uri}")
except Exception as e:
    print(f"Bronze write failed: {e}")
    traceback.print_exc()
    dbutils.notebook.exit(f"FAILED|{raw_count}|{str(e)}")

Bronze write complete → wasbs://bronze@etlworkflowautomation.blob.core.windows.net/chicago_crime/

In [0]:
# ─────────────────────────────────────────────
# 8. REGISTER AS DELTA TABLE (first run only)
# ─────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS chicago_crime")

# External tables don't support wasbs:// — use a view over the Delta path instead
spark.sql(f"""
    CREATE VIEW IF NOT EXISTS chicago_crime.bronze_crimes AS
    SELECT * FROM delta.`{bronze_uri}`
""")
print("✅ View 'chicago_crime.bronze_crimes' registered.")

✅ View 'chicago_crime.bronze_crimes' registered.

In [0]:
# ─────────────────────────────────────────────
# 9. RETURN STATUS TO ADF
# ─────────────────────────────────────────────
duration_sec = (datetime.now() - START_TIME).total_seconds()
print(f"Duration : {duration_sec:.1f}s  |  Rows written : {raw_count:,}")

# ADF reads this exit value via the notebook activity output
dbutils.notebook.exit(f"SUCCESS|{raw_count}|{duration_sec:.1f}")